## Interval Analysis

In [17]:
# !pip install tensorboardX
# !pip install bound-propagation

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter

use_cuda = False
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.MNIST('mnist_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.MNIST('mnist_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
bound_test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

## Simple NN. You can change this if you want. If you change it, mention the architectural details in your report.
class Net(nn.Sequential):
    def __init__(self):
        super(Net, self).__init__()
        self.fc = nn.Linear(28*28, 200)
        self.fc2 = nn.Linear(200,10)

    def forward(self, input):
        input = (input - 0.1307)/0.3081
        input = F.relu(self.fc(input))
        input = self.fc2(input)
        input = F.softmax(input, dim=-1) # added softmax for probabilities
        return input

# Add the data normalization as a first "layer" to the network
# this allows us to search for adverserial examples to the real image, rather than
# to the normalized image
model = Net()

model = model.to(device)
model.train()


Net(
  (fc): Linear(in_features=784, out_features=200, bias=True)
  (fc2): Linear(in_features=200, out_features=10, bias=True)
)

In [ ]:
def train_model(model, num_epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            images, labels = data
            images = images.view((-1, 28*28))
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.3f}')

def test_model(model):
    model.eval()
    
    with torch.no_grad():
        correct = 0
        total = 0
        for data in test_loader:
            images, labels = data
            images = images.view((-1, 28*28))
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f'Accuracy on images: {100 * correct / total}')
    

In [19]:
train_model(model, 15)

Epoch 1/15, Loss: 2.012
Epoch 2/15, Loss: 1.729
Epoch 3/15, Loss: 1.627
Epoch 4/15, Loss: 1.596
Epoch 5/15, Loss: 1.582
Epoch 6/15, Loss: 1.572
Epoch 7/15, Loss: 1.566
Epoch 8/15, Loss: 1.561
Epoch 9/15, Loss: 1.557
Epoch 10/15, Loss: 1.553
Epoch 11/15, Loss: 1.550
Epoch 12/15, Loss: 1.547
Epoch 13/15, Loss: 1.545
Epoch 14/15, Loss: 1.542
Epoch 15/15, Loss: 1.540


In [21]:
test_model(model)

Accuracy on images: 93.43


### Write the interval analysis for the simple model

In [10]:
## TODO: Write the interval analysis for the simple model
## you can use https://github.com/Zinoex/bound_propagation

from bound_propagation import BoundModelFactory, HyperRectangle

factory = BoundModelFactory()
isinstance(model, nn.Sequential)
bound_model = factory.build(model)

In [ ]:

bound_model.eval()
epsilons = np.linspace(0.01, 0.1, 10)
correct = 0
total = 0
for data in test_loader:
    images, labels = data
    images = images.view((-1, 28*28))
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)
    
    for epsilon in epsilons:
        input_bounds = HyperRectangle.from_eps(images, epsilon)
        ibp_bounds = bound_model.ibp(input_bounds) # type: ignore

        crown_bounds = bound_model.crown(input_bounds) # type: ignore
        crown_ibp_bounds = bound_model.crown_ibp(input_bounds) # type: ignore
        
        
        
    


In [12]:
from torch import nn
from bound_propagation import BoundModelFactory, HyperRectangle

class Network(nn.Sequential):
    def __init__(self):
        in_size = 30
        classes = 10

        super().__init__(
            nn.Linear(in_size, 16),
            nn.Tanh(),
            nn.Linear(16, 16),
            nn.Tanh(),
            nn.Linear(16, classes)
        )

net = Network()

factory = BoundModelFactory()
net = factory.build(net)

In [13]:
x = torch.rand(100, 30)
epsilon = 0.1
input_bounds = HyperRectangle.from_eps(x, epsilon)

ibp_bounds = net.ibp(input_bounds) # type: ignore

crown_bounds = net.crown(input_bounds).concretize() # type: ignore
crown_ibp_bounds = net.crown_ibp(input_bounds).concretize() # type: ignore

alpha_crown_bounds = net.crown(input_bounds, alpha=True).concretize() # type: ignore
alpha_crown_ibp_bounds = net.crown_ibp(input_bounds, alpha=True).concretize() # type: ignore

print("IBP bounds:", ibp_bounds.lower, ibp_bounds.upper)
print("CROWN bounds:", crown_bounds.lower, crown_bounds.upper)
print("CROWN-IBP bounds:", crown_ibp_bounds.lower, crown_ibp_bounds.upper)
print("Alpha CROWN bounds:", alpha_crown_bounds.lower, alpha_crown_bounds.upper)
print("Alpha CROWN-IBP bounds:", alpha_crown_ibp_bounds.lower, alpha_crown_ibp_bounds.upper)


IBP bounds: tensor([[-1.0705, -1.0305, -0.9168, -1.1060, -0.8685, -0.8150, -1.2156, -0.7391,
         -1.0446, -0.5084],
        [-1.0680, -1.0226, -0.8680, -1.0500, -0.8293, -0.8362, -1.2056, -0.7178,
         -1.0333, -0.5322],
        [-1.0600, -0.9998, -0.8805, -1.0853, -0.8682, -0.8341, -1.2158, -0.7451,
         -1.0572, -0.5248],
        [-1.0516, -1.0072, -0.8684, -1.0596, -0.8456, -0.7933, -1.1987, -0.7272,
         -1.0243, -0.5116],
        [-1.0238, -1.0032, -0.9615, -1.0640, -0.8215, -0.8747, -1.1817, -0.7253,
         -1.0618, -0.5135],
        [-1.0295, -1.0250, -0.8755, -0.9831, -0.7947, -0.7966, -1.1683, -0.6818,
         -0.9970, -0.4923],
        [-0.9705, -1.0673, -0.9441, -1.0589, -0.8402, -0.8017, -1.1847, -0.6734,
         -1.0169, -0.4384],
        [-1.0067, -1.1020, -0.9986, -1.1071, -0.8404, -0.8801, -1.1887, -0.6980,
         -1.0544, -0.4719],
        [-0.9831, -1.0613, -0.9024, -1.0006, -0.8027, -0.8392, -1.1822, -0.7010,
         -1.0191, -0.4758],
       